<h1> Funções de Banco de Dados Úteis </h1>

<h4> Importação de Pacotes e Bibliotecas </h4>

In [4]:
import sqlite3

<h4> Definição de Métodos </h4>

Conexão com banco de dados

In [3]:
DB_NAME = "ifood_app.db"
def get_connection():
    try:
        conn = sqlite3.connect(DB_NAME)
        conn.row_factory = sqlite3.Row
    except sqlite3.Error as e:
        print(f"Erro ao conectar ao banco de dados: {e}")
        return None
    else:
        print("Banco de Dados conectado com sucesso.")
        return conn

<h3> CRUD PRODUTO </h3>

Criando tabelas

In [ ]:
def criar_tabelas():
    try:
        conn = get_connection()
        cursor = conn.cursor()

        cursor.execute("""
        CREATE TABLE IF NOT EXISTS produto (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            nome TEXT NOT NULL,
            categoria TEXT NOT NULL,
            url_foto TEXT,
            preco REAL NOT NULL
        )
        """)

        cursor.execute("""
        CREATE TABLE IF NOT EXISTS pedido (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            data_hora DATETIME DEFAULT CURRENT_TIMESTAMP,
            preco_total REAL
        )
        """)

        cursor.execute("""
        CREATE TABLE IF NOT EXISTS pedido_produto (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            pedido_id INTEGER,
            produto_id INTEGER,
            quantidade INTEGER,
            FOREIGN KEY (pedido_id) REFERENCES pedido(id),
            FOREIGN KEY (produto_id) REFERENCES produto(id)
        )
        """)

        cursor.execute("""
        CREATE TABLE IF NOT EXISTS feedback (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            pedido_id INTEGER,
            produto_id INTEGER,
            nome_cliente TEXT,
            descricao TEXT,
            sentimento_final TEXT,
            FOREIGN KEY (pedido_id) REFERENCES pedido(id),
            FOREIGN KEY (produto_id) REFERENCES produto(id)
        )
        """)

        conn.commit()
        conn.close()
    except sqlite3.Error as e:
        print(f"Erro ao criar tabelas: {e}")
    else:
        print(f"Todas as tabelas foram criadas com sucesso no banco de dados {DB_NAME}.")

Criação do produto

Listar Produtos

Atualizar produto

In [23]:
def atualizar_produto(id, nome, categoria, url_foto, preco):
    try:    
        conn = get_connection()
        cursor = conn.cursor()

        cursor.execute("""
        UPDATE produto
        SET nome = ?, categoria = ?, url_foto = ?, preco = ?
        WHERE id = ?
        """, (nome, categoria, url_foto, preco, id))

        conn.commit()
        conn.close()
    except sqlite3.Error as e:
        print(f"Erro ao atualizar produto: {e}")    
    else:
        print(f"Produto com ID {id} atualizado com sucesso.")

Excluir produto

In [24]:
def deletar_produto(id):
    try:
        conn = get_connection()
        cursor = conn.cursor()

        cursor.execute("DELETE FROM produto WHERE id = ?", (id,))

        conn.commit()
        conn.close()
    except sqlite3.Error as e:
        print(f"Erro ao deletar produto: {e}")  
    else:
        print(f"Produto com ID {id} deletado com sucesso.")

<h3> CRUD PEDIDO </h3>

Criar pedido

In [25]:
def criar_pedido():
    try:
        conn = get_connection()
        cursor = conn.cursor()

        cursor.execute("""
        INSERT INTO pedido (preco_total)
        VALUES (0)
        """)

        pedido_id = cursor.lastrowid

        conn.commit()
        conn.close()

    except sqlite3.Error as e:
        print(f"Erro ao criar pedido: {e}")
        return None
    else:
        print(f"Pedido criado com sucesso com ID {pedido_id}.")
        return pedido_id    

Listar pedido

In [26]:
def listar_pedidos():
    try:
        conn = get_connection()
        cursor = conn.cursor()

        cursor.execute("SELECT * FROM pedido")

        pedidos = cursor.fetchall()

        conn.close()
        
    except sqlite3.Error as e:
        print(f"Erro ao listar pedidos: {e}")
        return []   
    else:
        print(f"{len(pedidos)} pedidos listados com sucesso.")
        return pedidos

Atualizar preco pedido

In [27]:
def atualizar_preco_pedido(pedido_id):

    conn = get_connection()
    cursor = conn.cursor()

    cursor.execute("""
    UPDATE pedido
    SET preco_total = (
        SELECT SUM(pp.quantidade * p.preco)
        FROM pedido_produto pp
        JOIN produto p ON pp.produto_id = p.id
        WHERE pp.pedido_id = ?
    )
    WHERE id = ?
    """, (pedido_id, pedido_id))

    conn.commit()
    conn.close()

<h3> CRUD PEDIDO_PRODUTO </h3>

Adicionar produto pedido

In [28]:
def adicionar_produto_pedido(pedido_id, produto_id, quantidade):
    try:
        conn = get_connection()
        cursor = conn.cursor()

        cursor.execute("""
        INSERT INTO pedido_produto (pedido_id, produto_id, quantidade)
        VALUES (?, ?, ?)
        """, (pedido_id, produto_id, quantidade))

        conn.commit()

        conn.close()

        atualizar_preco_pedido(pedido_id)
    except sqlite3.Error as e:
        print(f"Erro ao adicionar produto ao pedido: {e}")  
    else:
        print(f"Produto com ID {produto_id} adicionado ao pedido com ID {pedido_id} com quantidade {quantidade}.")
        


Listar produtos de pedidos

In [29]:
def listar_produtos_pedido(pedido_id):
    try:
        conn = get_connection()
        cursor = conn.cursor()

        cursor.execute("""
        SELECT
            p.nome,
            p.preco,
            pp.quantidade
        FROM pedido_produto pp
        JOIN produto p
        ON pp.produto_id = p.id
        WHERE pp.pedido_id = ?
        """, (pedido_id,))

        dados = cursor.fetchall()

        conn.close()

        return dados
    except sqlite3.Error as e:
        print(f"Erro ao listar produtos do pedido: {e}")
        return []
    else:
        print(f"{len(dados)} produtos listados para o pedido com ID {pedido_id}.")

<h3> CRUD FEEDBACK </h3>

Criar feedback

In [ ]:
'''
def criar_feedback(pedido_id, produto_id, nome_cliente, descricao, categoria,sentimento):
    try:
        conn = get_connection()
        cursor = conn.cursor()

        cursor.execute("""
        INSERT INTO feedback
        (pedido_id, produto_id, nome_cliente, descricao, categoria, sentimento_final)
        VALUES (?, ?, ?, ?, ?, ?)
        """, (pedido_id, produto_id, nome_cliente, descricao, categoria, sentimento))

        conn.commit()
        conn.close()
    except sqlite3.Error as e:
        print(f"Erro ao criar feedback: {e}")   
    else:
        print(f"Feedback criado com sucesso para o pedido ID {pedido_id} e produto ID {produto_id}.")
'''

Listar Feedback

In [31]:
def listar_feedbacks():
    try:
        conn = get_connection()
        cursor = conn.cursor()

        cursor.execute("""
        SELECT
            f.id,
            f.nome_cliente,
            p.nome as produto,
            f.descricao,
            f.sentimento_final
        FROM feedback f
        JOIN produto p
        ON f.produto_id = p.id
        """)

        dados = cursor.fetchall()

        conn.close()
    except sqlite3.Error as e:
        print(f"Erro ao listar feedbacks: {e}")
        return []
    else:        
        print(f"{len(dados)} feedbacks listados com sucesso.")
        return dados

Excluir Feedback

In [32]:
def deletar_feedback(id):
    try:
        conn = get_connection()
        cursor = conn.cursor()

        cursor.execute("DELETE FROM feedback WHERE id = ?", (id,))

        conn.commit()
        conn.close()
    except sqlite3.Error as e:
        print(f"Erro ao deletar feedback: {e}")
    else:
        print(f"Feedback com ID {id} deletado com sucesso.")

<h2> Exemplos de inserções. Exemplos </h2>

In [5]:
criar_tabelas()

Banco de Dados conectado com sucesso.
Todas as tabelas foram criadas com sucesso no banco de dados ifood_app.db.


In [8]:
info = listar_tabelas_e_colunas()
print(f"Qtde de tabelas: {len(info)}")
for tabela, colunas in info.items():
    print(f"Tabela: {tabela}")
    for c in colunas:
        print(f"  - {c['nome']} ({c['tipo']}) PK={c['pk']} NOT NULL={c['notnull']}")
        print()

Qtde de tabelas: 4
Tabela: produto
  - id (INTEGER) PK=1 NOT NULL=0

  - nome (TEXT) PK=0 NOT NULL=1

  - categoria (TEXT) PK=0 NOT NULL=1

  - url_foto (TEXT) PK=0 NOT NULL=0

  - preco (REAL) PK=0 NOT NULL=1

Tabela: pedido
  - id (INTEGER) PK=1 NOT NULL=0

  - data_hora (DATETIME) PK=0 NOT NULL=0

  - preco_total (REAL) PK=0 NOT NULL=0

Tabela: pedido_produto
  - id (INTEGER) PK=1 NOT NULL=0

  - pedido_id (INTEGER) PK=0 NOT NULL=0

  - produto_id (INTEGER) PK=0 NOT NULL=0

  - quantidade (INTEGER) PK=0 NOT NULL=0

Tabela: feedback
  - id (INTEGER) PK=1 NOT NULL=0

  - pedido_id (INTEGER) PK=0 NOT NULL=0

  - produto_id (INTEGER) PK=0 NOT NULL=0

  - nome_cliente (TEXT) PK=0 NOT NULL=0

  - descricao (TEXT) PK=0 NOT NULL=0

  - sentimento_final (TEXT) PK=0 NOT NULL=0



In [44]:
# Exemplos de produtos inseridos
produtos_exemplo = [
    ("Banana", "frutas", "https://example.com/banana.jpg", 5.50),
    ("Maçã", "frutas", "https://example.com/maca.jpg", 7.00),
    ("Laranja", "frutas", "https://example.com/laranja.jpg", 6.00),
    ("Alface", "verduras", "https://example.com/alface.jpg", 4.50),
    ("Tomate", "verduras", "https://example.com/tomate.jpg", 5.20),
    ("Cenoura", "verduras", "https://example.com/cenoura.jpg", 3.80),
    ("Leite Integral", "laticinios", "https://example.com/leite.jpg", 6.90),
    ("Queijo Mussarela", "laticinios", "https://example.com/queijo.jpg", 18.50),
    ("Pão Francês", "padaria", "https://example.com/pao.jpg", 0.80),
    ("Sabão em Pó", "limpeza", "https://example.com/sabao.jpg", 18.90)
]
# Inserindo os produtos no banco
for p in produtos_exemplo:
    criar_produto(*p)

Banco de Dados conectado com sucesso.
Produto 'Banana' criado com sucesso.
Banco de Dados conectado com sucesso.
Produto 'Maçã' criado com sucesso.
Banco de Dados conectado com sucesso.
Produto 'Laranja' criado com sucesso.
Banco de Dados conectado com sucesso.
Produto 'Alface' criado com sucesso.
Banco de Dados conectado com sucesso.
Produto 'Tomate' criado com sucesso.
Banco de Dados conectado com sucesso.
Produto 'Cenoura' criado com sucesso.
Banco de Dados conectado com sucesso.
Produto 'Leite Integral' criado com sucesso.
Banco de Dados conectado com sucesso.
Produto 'Queijo Mussarela' criado com sucesso.
Banco de Dados conectado com sucesso.
Produto 'Pão Francês' criado com sucesso.
Banco de Dados conectado com sucesso.
Produto 'Sabão em Pó' criado com sucesso.


In [46]:
#Listagem de produtos
item = listar_produtos()
for item in item:
    print(f"ID: {item['id']}, Nome: {item['nome']}, Categoria: {item['categoria']}, Preço: {item['preco']}")

Banco de Dados conectado com sucesso.
10 produtos listados com sucesso.
ID: 1, Nome: Banana, Categoria: frutas, Preço: 5.5
ID: 2, Nome: Maçã, Categoria: frutas, Preço: 7.0
ID: 3, Nome: Laranja, Categoria: frutas, Preço: 6.0
ID: 4, Nome: Alface, Categoria: verduras, Preço: 4.5
ID: 5, Nome: Tomate, Categoria: verduras, Preço: 5.2
ID: 6, Nome: Cenoura, Categoria: verduras, Preço: 3.8
ID: 7, Nome: Leite Integral, Categoria: laticinios, Preço: 6.9
ID: 8, Nome: Queijo Mussarela, Categoria: laticinios, Preço: 18.5
ID: 9, Nome: Pão Francês, Categoria: padaria, Preço: 0.8
ID: 10, Nome: Sabão em Pó, Categoria: limpeza, Preço: 18.9


In [ ]:
pedido_id = criar_pedido()

adicionar_produto_pedido(pedido_id, 1, 3)

criar_feedback(pedido_id, 1, "Carlos", "Muito bom", "positivo")

print(listar_feedbacks())

<h2> Resultado Final </h2>

In [1]:
def criar_tabelas_banco_dados():
    try:
        conn = get_connection()
        cursor = conn.cursor()

        # Tabela produto
        cursor.execute("""
        CREATE TABLE IF NOT EXISTS produto (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            nome TEXT NOT NULL,
            categoria TEXT NOT NULL,
            url_foto TEXT,
            preco REAL NOT NULL
        )
        """)


        # Criar tabela feedback_temporario
        cursor.execute("""
        CREATE TABLE IF NOT EXISTS feedback_temporario (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            nome_cliente TEXT,
            descricao TEXT,
            sentimento_final TEXT,
            categoria TEXT,
            data_hora DATETIME DEFAULT CURRENT_TIMESTAMP,
            produto_id INTEGER,
            FOREIGN KEY (produto_id) REFERENCES produto(id)
        )
        """)

        conn.commit()
        conn.close()

    except sqlite3.Error as e:
        print(f"Erro ao criar tabelas: {e}")
    else:
        print(f"Tabelas criadas com sucesso.")

In [23]:
def get_schema_overview_grouped():
    conn = get_connection()
    cursor = conn.cursor()

    query = """
    SELECT 
        m.name AS tabela,
        p.name AS campo,
        p.type AS tipo
    FROM sqlite_master m
    JOIN pragma_table_info(m.name) p
    WHERE m.type = 'table'
    ORDER BY m.name, p.cid;
    """

    cursor.execute(query)
    rows = cursor.fetchall()
    conn.close()

    schema = {}
    
    for tabela, campo, tipo in rows:
        # ignorar tabelas internas do SQLite
        if tabela.startswith("sqlite_"):
            continue

        if tabela not in schema:
            schema[tabela] = {
                "qtd_campos": 0,
                "campos": []
            }

        schema[tabela]["campos"].append({
            "campo": campo,
            "tipo": tipo
        })
        schema[tabela]["qtd_campos"] += 1

    result = {
        "qtd_total_tabelas": len(schema),
        "tabelas": schema
    }

    return result

In [25]:
def limpar_todas_tabelas():
    try:
        conn = get_connection()
        cursor = conn.cursor()

        # pega todas as tabelas do banco
        cursor.execute("""
            SELECT name 
            FROM sqlite_master 
            WHERE type='table'
            AND name NOT LIKE 'sqlite_%';
        """)

        tabelas = cursor.fetchall()

        # desabilita FK temporariamente (evita erro de relacionamento)
        cursor.execute("PRAGMA foreign_keys = OFF;")

        for (tabela,) in tabelas:
            cursor.execute(f"DELETE FROM {tabela};")

        conn.commit()

        # reativa FK
        cursor.execute("PRAGMA foreign_keys = ON;")

        conn.close()

        print("Todas as tabelas foram limpas com sucesso.")

    except sqlite3.Error as e:
        print(f"Erro ao limpar tabelas: {e}")

In [26]:
limpar_todas_tabelas()

Banco de Dados conectado com sucesso.
Todas as tabelas foram limpas com sucesso.


In [ ]:
criar_tabelas_banco_dados()


Banco de Dados conectado com sucesso.
Tabelas criadas com sucesso.


In [24]:
get_schema_overview_grouped()

Banco de Dados conectado com sucesso.


{'qtd_total_tabelas': 2,
 'tabelas': {'feedback_temporario': {'qtd_campos': 7,
   'campos': [{'campo': 'id', 'tipo': 'INTEGER'},
    {'campo': 'nome_cliente', 'tipo': 'TEXT'},
    {'campo': 'descricao', 'tipo': 'TEXT'},
    {'campo': 'sentimento_final', 'tipo': 'TEXT'},
    {'campo': 'categoria', 'tipo': 'TEXT'},
    {'campo': 'data_hora', 'tipo': 'DATETIME'},
    {'campo': 'produto_id', 'tipo': 'INTEGER'}]},
  'produto': {'qtd_campos': 5,
   'campos': [{'campo': 'id', 'tipo': 'INTEGER'},
    {'campo': 'nome', 'tipo': 'TEXT'},
    {'campo': 'categoria', 'tipo': 'TEXT'},
    {'campo': 'url_foto', 'tipo': 'TEXT'},
    {'campo': 'preco', 'tipo': 'REAL'}]}}}

In [ ]:
def criar_produto(nome, categoria, url_foto, preco):
    try:
        conn = get_connection()
        cursor = conn.cursor()

        cursor.execute("""
        INSERT INTO produto (nome, categoria, url_foto, preco)
        VALUES (?, ?, ?, ?)
        """, (nome, categoria, url_foto, preco))

        conn.commit()
        conn.close()
    except sqlite3.Error as e:
        print(f"Erro ao criar produto: {e}")
    else:
        print(f"Produto '{nome}' criado com sucesso.")

In [27]:
def listar_produtos():
    try:
        conn = get_connection()
        cursor = conn.cursor()
        cursor.execute("SELECT * FROM produto")
        produtos = cursor.fetchall()
        conn.close()
    except sqlite3.Error as e:
        print(f"Erro ao listar produtos: {e}")
        return []
    else:
        print(f"{len(produtos)} produtos listados com sucesso.")
        return produtos

In [30]:
# CRUD FEEDBACK TEMPORÁRIO - SEM VÍNCULO COM PEDIDO
# CATEGORIA: Principais categorias com SCORE
def criar_feedback(nome_cliente, descricao, sentimento_final, categoria,produto_id):
    try:
        conn = get_connection()
        cursor = conn.cursor()

        cursor.execute("""
            INSERT INTO feedback_temporario
            (nome_cliente, descricao, sentimento_final, categoria,produto_id)
            VALUES (?, ?, ?, ?)
        """, (nome_cliente, descricao, sentimento_final,categoria,produto_id))

        conn.commit()
        conn.close()

    except sqlite3.Error as e:
        print(f"Erro ao cadastrar feedback: {e}")

    else:
        print("Feedback cadastrado com sucesso.")

In [29]:
def listar_feedbacks():
    try:
        conn = get_connection()
        conn.row_factory = sqlite3.Row
        cursor = conn.cursor()

        cursor.execute("""
            SELECT *
            FROM feedback_temporario
            ORDER BY data_hora DESC
        """)

        feedbacks = cursor.fetchall()

        conn.close()

        return feedbacks

    except sqlite3.Error as e:
        print(f"Erro ao listar feedbacks: {e}")
        return []

In [31]:
listar_feedbacks()

Banco de Dados conectado com sucesso.


[]

In [ ]:
def listar_feedbacks_com_produto():
    try:
        conn = get_connection()
        conn.row_factory = sqlite3.Row
        cursor = conn.cursor()

        cursor.execute("""
            SELECT 
                f.id,
                f.nome_cliente,
                f.descricao,
                f.sentimento_final,
                f.categoria,
                f.data_hora,
                p.nome AS produto
            FROM feedback_temporario f
            LEFT JOIN produto p
            ON f.produto_id = p.id
            ORDER BY f.data_hora DESC
        """)

        feedbacks = cursor.fetchall()

        conn.close()

        return feedbacks

    except sqlite3.Error as e:
        print(f"Erro ao listar feedbacks: {e}")
        return []